# Post-simulation analysis for protein MD

This notebook is the companion to ``protein-simulation-setup.ipynb`` and assumes a production trajectory has already been generated.

It performs:
* energy sanity checks from the ``.edr`` file,
* trajectory alignment and PBC imaging,
* interactive geometric measurements (distances, angles, dihedrals),
* secondary-structure timeline (DSSP),
* Ramachandran analysis,
* and exports a clean, aligned trajectory/topology pair for visualisation in **Molstar**.

All plots are saved to the ``analysis_output/`` directory in publication-ready format.

## Configuration

Set the same ``name`` you used in the setup notebook, then derived file paths.
Adjust ``timestep_ps`` if your integration step differs from the standard 2 fs.

In [ ]:
name = 'protein'

traj_path  = name + '.xtc'
top_path   = name + '.gro'
edr_path   = name + '.edr'
tpr_path   = name + '.tpr'

timestep_ps = 2.0          # integration step in ps
out_dir     = 'analysis_output'

## Imports

In [ ]:
from pathlib import Path

import mdtraj as md
import nglview as nv
import numpy as np
from matplotlib import pyplot as plt

import mdanalysis_utils as mdu

## Energy sanity checks

Parse the GROMACS ``.edr`` energy file and plot temperature, pressure, and potential energy versus time.
We rely on ``gmx energy`` to write xvg files, then load the data with NumPy.

In [ ]:
import gromacs as gmx

Path(out_dir).mkdir(parents=True, exist_ok=True)

# Extract energy terms to xvg files.
gmx.energy(f=edr_path, o=str(Path(out_dir) / 'temp.xvg'),  input='Temperature')
gmx.energy(f=edr_path, o=str(Path(out_dir) / 'press.xvg'), input='Pressure')
gmx.energy(f=edr_path, o=str(Path(out_dir) / 'pot.xvg'),   input='Potential')

In [ ]:
def _load_xvg(path: Path) -> None:
    """Load an xvg file, skipping Gracecomment lines."""
    return np.loadtxt(path, comments=['#', '@'])

temp  = _load_xvg(Path(out_dir) / 'temp.xvg')
press = _load_xvg(Path(out_dir) / 'press.xvg')
pot   = _load_xvg(Path(out_dir) / 'pot.xvg')

mdu.plot_energy(
    temp[:, 0] / 1000.0, temp[:, 1],
    title='Temperature', ylabel='Temperature (K)',
    outpath=Path(out_dir) / 'temperature.png'
)
mdu.plot_energy(
    press[:, 0] / 1000.0, press[:, 1],
    title='Pressure', ylabel='Pressure (bar)',
    outpath=Path(out_dir) / 'pressure.png'
)
mdu.plot_energy(
    pot[:, 0] / 1000.0, pot[:, 1],
    title='Potential energy', ylabel='Potential energy (kJ/mol)',
    outpath=Path(out_dir) / 'potential.png'
)

## Trajectory preprocessing

Load the raw trajectory, **align** the protein to the first frame, and attempt
**PBC imaging** so that molecules are whole.  A quick ``nglview`` preview lets you
inspect the aligned trajectory inline.

In [ ]:
traj = md.load(traj_path, top=top_path)
print(f'Loaded {traj.n_frames} frames, {traj.n_atoms} atoms, {traj.n_residues} residues')

In [ ]:
traj_aligned = mdu.align_trajectory(traj)
traj_aligned = mdu.image_molecules(traj_aligned)
print('Alignment and imaging complete.')

In [ ]:
nv.show_mdtraj(traj_aligned)

## Geometric measurements

Measure distances, angles, and dihedrals between arbitrary atom selections.
Selections use MDTraj's DSL (e.g. ``"residue 10 and name CA"``).

Fill in the lists below with your selections of interest.  Empty lists are fine
— the corresponding plots will simply be skipped.

In [ ]:
# Distance pairs: each tuple is (selection_a, selection_b)
distance_pairs = [
    # Example: ("residue 10 and name CA", "residue 20 and name CA"),
]

# Angle triples: each tuple is (sel_a, sel_b, sel_c) for angle a-b-c
angle_triples = [
    # Example: ("residue 10 and name CA", "residue 11 and name CA", "residue 12 and name CA"),
]

# Dihedral quads: each tuple is (sel_a, sel_b, sel_c, sel_d)
dihedral_quads = [
    # Example: ("residue 10 and name N", "residue 10 and name CA", "residue 10 and name C", "residue 11 and name N"),
]

In [ ]:
if distance_pairs:
    t, dists, labels = mdu.measure_distance(traj_aligned, distance_pairs, timestep_ps=timestep_ps)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    for i, label in enumerate(labels):
        ax.plot(t, dists[:, i], label=label, linewidth=0.8)
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Distance (nm)')
    ax.set_title('Pairwise distances')
    ax.legend(fontsize=8, frameon=False)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.7)
    fig.savefig(Path(out_dir) / 'distances.png')
    plt.show()

In [ ]:
if angle_triples:
    t, angles, labels = mdu.measure_angle(traj_aligned, angle_triples, timestep_ps=timestep_ps)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    for i, label in enumerate(labels):
        ax.plot(t, angles[:, i], label=label, linewidth=0.8)
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Angle (°)')
    ax.set_title('Angles')
    ax.legend(fontsize=8, frameon=False)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.7)
    fig.savefig(Path(out_dir) / 'angles.png')
    plt.show()

In [ ]:
if dihedral_quads:
    t, dihs, labels = mdu.measure_dihedral(traj_aligned, dihedral_quads, timestep_ps=timestep_ps)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    for i, label in enumerate(labels):
        ax.plot(t, dihs[:, i], label=label, linewidth=0.8)
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Dihedral (°)')
    ax.set_title('Dihedrals')
    ax.legend(fontsize=8, frameon=False)
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.7)
    fig.savefig(Path(out_dir) / 'dihedrals.png')
    plt.show()

## Structure quality — DSSP timeline

Run DSSP on every frame and display a heat-map of secondary-structure evolution.
Helices, sheets, and coil/other are shown on a colour scale.

In [ ]:
time_ns, residues, ss_codes = mdu.dssp_timeline(
    traj_aligned,
    outpath=Path(out_dir) / 'dssp_timeline.png'
)
print(f'DSSP computed for {traj_aligned.n_frames} frames x {len(residues)} residues')

## Structure quality — Ramachandran plot

Extract backbone φ / ψ dihedrals for every frame and draw a 2-D histogram.
If you want to restrict the analysis to specific residues, supply a list of
0-based residue indices to ``residue_indices``.

In [ ]:
phi, psi = mdu.ramachandran(
    traj_aligned,
    residue_indices=None,  # set to e.g. [10, 11, 12] to zoom in
    outpath=Path(out_dir) / 'ramachandran.png'
)
print(f'Collected {len(phi)} (phi, psi) measurements')

## Molstar-aligned file export

Save the aligned topology and trajectory so they can be loaded directly into
**Molstar** for high-quality visualisation.

In [ ]:
aligned_gro = Path(out_dir) / f'{name}-aligned.gro'
aligned_xtc = Path(out_dir) / f'{name}-aligned.xtc'

mdu.save_aligned_files(traj_aligned, str(aligned_gro), str(aligned_xtc))
print(f'Aligned files saved to {out_dir}/')
print(f'  Structure : {aligned_gro}')
print(f'  Trajectory: {aligned_xtc}')